In [1]:
import autogen
import chromadb
from chromadb.utils import embedding_functions
import uuid
from datetime import datetime

class VectorMemory:
    def __init__(self, collection_name="agent_memories"):
        self.client = chromadb.Client()
        self.embedding_fn = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.create_collection(
            name=collection_name,
            embedding_function=self.embedding_fn
        )

    def store(self, content: str, context: str = ""):
        self.collection.add(
            documents=[content],
            metadatas=[{"context": context, "timestamp": str(datetime.now())}],
            ids=[str(uuid.uuid4())]
        )

    def retrieve(self, query: str, n_results: int = 3):
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results['documents'][0]

class AgentWithMemory:
    def __init__(self):
        self.memory = VectorMemory()
        self.config_list = [{"model": "gpt-4", "api_key": "your-key-here"}]
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            llm_config={
                "config_list": self.config_list,
                "cache_seed": 42
            }
        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1
        )

    def chat(self, message: str):
        # Retrieve relevant memories
        memories = self.memory.retrieve(message)
        context = f"Previous relevant information: {memories}\n\nCurrent query: {message}"
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context
        )
        
        # Store new memory
        self.memory.store(
            content=self.assistant.last_message()["content"],
            context=message
        )
        
        return self.assistant.last_message()["content"]

d:\GitHub\AutoGen-notebooks\.venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
import autogen
import chromadb
import openai
from typing import List, Dict
import uuid
from datetime import datetime

class VectorMemory:
    def __init__(self, openai_api_key: str, collection_name: str = "agent_memories"):
        self.client = chromadb.Client()
        self.collection = self.client.create_collection(name=collection_name)
        openai.api_key = openai_api_key
        
    def _get_embedding(self, text: str) -> List[float]:
        response = openai.Embedding.create(
            input=text,
            model="text-embedding-ada-002"
        )
        return response['data'][0]['embedding']
    
    def store(self, content: str, context: str = ""):
        embedding = self._get_embedding(content)
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "timestamp": str(datetime.now())
            }],
            ids=[str(uuid.uuid4())]
        )
    
    def retrieve(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = self._get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]

class AutoGenWithMemory:
    def __init__(self, openai_api_key: str):
        self.memory = VectorMemory(openai_api_key)
        self.config_list = [{
            "model": "gpt-4",
            "api_key": openai_api_key
        }]
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            llm_config={
                "config_list": self.config_list,
                "cache_seed": 42
            }
        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER",
            max_consecutive_auto_reply=1
        )
    
    def chat(self, message: str) -> str:
        # Retrieve relevant memories
        memories = self.memory.retrieve(message)
        context = f"Previous relevant context:\n{memories}\n\nCurrent query: {message}"
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context
        )
        
        response = self.assistant.last_message()["content"]
        
        # Store new memory
        self.memory.store(content=response, context=message)
        
        return response

In [ ]:

class OllamaEmbedding:
    # def __init__(self, base_url: str = "http://localhost:11434"):
    #     self.base_url = base_url
        
    def get_embedding(self, text: str) -> List[float]:
        response = ollama.embeddings(model="nomic-embed-text",prompt=text)
        return response['embedding']

class VectorMemory:
    def __init__(self, collection_name: str = "agent_memories"):
        self.client = chromadb.PersistentClient(path="./chroma_db")
        # Get collection if exists, create if it doesn't
        try:
            self.collection = self.client.get_collection(name=collection_name)
        except:
            self.collection = self.client.create_collection(name=collection_name)
        self.embedder = OllamaEmbedding()
    
    def store(self, content: str, context: str = ""):
        embedding = self.embedder.get_embedding(content)
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "timestamp": str(datetime.now())
            }],
            ids=[str(uuid.uuid4())]
        )
    
    def retrieve(self, query: str, n_results: int = 3) -> List[str]:
        query_embedding = self.embedder.get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )
        return results['documents'][0]


In [1]:
import autogen
import chromadb
import requests
import numpy as np
from typing import List, Dict, Optional
import uuid
from datetime import datetime
import ollama


class OllamaEmbedding:
    # def __init__(self, base_url: str = "http://localhost:11434"):
    #     self.base_url = base_url
        
    def get_embedding(self, text: str) -> List[float]:
        response = ollama.embeddings(model="nomic-embed-text",prompt=text)
        return response['embedding']

class VectorMemory:
    def __init__(self, collection_name: str = "agent_memories"):
        self.client = chromadb.PersistentClient(path="./chroma_db")
        try:
            self.collection = self.client.get_collection(name=collection_name)
        except:
            self.collection = self.client.create_collection(name=collection_name)
        self.embedder = OllamaEmbedding()

    def create_conversation(self) -> str:
        """Create new conversation ID"""
        return str(uuid.uuid4())

    def store(self, content: str, context: str = "", conversation_id: Optional[str] = None) -> str:
        """Store message with conversation ID"""
        if not conversation_id:
            conversation_id = self.create_conversation()
            
        embedding = self.embedder.get_embedding(content)
        message_id = str(uuid.uuid4())
        
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "conversation_id": conversation_id,
                "timestamp": str(datetime.now()),
                "message_id": message_id
            }],
            ids=[message_id]
        )
        return conversation_id

    def get_conversation(self, conversation_id: str) -> List[Dict]:
        """Retrieve all messages in a conversation"""
        results = self.collection.get(
            where={"conversation_id": conversation_id},
            include=["documents", "metadatas"]
        )
        
        # Sort by timestamp
        conversation = list(zip(results['documents'], results['metadatas']))
        conversation.sort(key=lambda x: x[1]['timestamp'])
        return conversation

    def retrieve(self, query: str, n_results: int = 3, conversation_id: Optional[str] = None) -> List[str]:
        """Retrieve similar messages, optionally filtered by conversation"""
        query_embedding = self.embedder.get_embedding(query)
        
        where_clause = {"conversation_id": conversation_id} if conversation_id else None
        
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_clause
        )
        return results['documents'][0]

class AutoGenWithMemory:
    def __init__(self, config_list: List[Dict]):
        self.memory = VectorMemory()
        
        self.assistant = autogen.AssistantAgent(
            name="assistant",
            system_message="You are helpfil assistant to help user with their queries.",
            llm_config={
                "config_list": config_list,
                "cache_seed": 42
            },
            code_execution_config=False

        )
        
        self.user_proxy = autogen.UserProxyAgent(
            name="user_proxy",
            human_input_mode="NEVER", 
            max_consecutive_auto_reply=1,
            code_execution_config=False,
            is_termination_msg= lambda msg: "TERMINATE" in msg["content"],
        )
    
    def chat(self, message: str, msg_id:str) -> str:
        # Get relevant memories
        if msg_id:
            memories = self.memory.retrieve(message, conversation_id=msg_id)
            # Build context
            context = (
            "Previous relevant information:\n"
            + '\n'.join(memories) + "\n\n"
            + f"Current query: {message}")
        else:
            context = f"Current query: {message}"
        
        
        
        # Chat with context
        self.user_proxy.initiate_chat(
            self.assistant,
            message=context + " and Say the word TERMINATE."
        )
        
        response = self.assistant.last_message()["content"]
        
        # Store new memory
        id=self.memory.store(content=response, context=message)
        
        return response , id

d:\GitHub\AutoGen-notebooks\.venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [2]:
# from vector_memory import AutoGenWithMemory

# Configuration
config_list = [
    {
        "model": "llama3.2",
        "base_url": "http://localhost:11434/v1",
        'api_key': 'ollama',
    },
]

# Initialize agent
agent = AutoGenWithMemory(config_list)

# Example conversation
response1 = agent.chat("My name is anoop, I like pizaa", None)
print("Response 1:", response1)



user_proxy (to assistant):

Current query: My name is anoop, I like pizaa and Say the word TERMINATE.

--------------------------------------------------------------------------------
[autogen.oai.client: 12-25 22:54:08] {432} WARNING - Model llama3.2 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to user_proxy):

Hello Anoop!

It sounds like you enjoy pizza, and I'm here to help you with anything on your mind. However, you also mentioned a specific word that might be important to talk about.

I want to acknowledge that the word "TERMINATE" can have different meanings in various contexts. Can you please clarify or provide more information about why this word is significant to you? Is it related to something you'd like to discuss, a goal you're working towards, or perhaps something else?

Let's chat about pizza or anything else you'd like to talk about, and I'll be happy

In [3]:
# Follow-up with memory
response2 = agent.chat("What is my name?, What do I like?","d02c8e83-0eec-4a67-aaea-70c32f051e42")
print("Response 2:", response2)

user_proxy (to assistant):

Previous relevant information:
Hello Anoop!

It sounds like you enjoy pizza, and I'm here to help you with anything on your mind. However, you also mentioned a specific word that might be important to talk about.

I want to acknowledge that the word "TERMINATE" can have different meanings in various contexts. Can you please clarify or provide more information about why this word is significant to you? Is it related to something you'd like to discuss, a goal you're working towards, or perhaps something else?

Let's chat about pizza or anything else you'd like to talk about, and I'll be happy to help!

Current query: What is my name?, What do I like? and Say the word TERMINATE.

--------------------------------------------------------------------------------


[autogen.oai.client: 12-25 22:55:44] {432} WARNING - Model llama3.2 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
assistant (to user_proxy):

Hello again! Don't worry, we were in the middle of a conversation that didn't quite follow the usual flow.

To answer your question simply: You are not telling me what you like (if that's something I can help with), but you're asking about "TERMINATE". If I recall correctly from our previous conversation, you might be thinking about a word that was mentioned in passing, and it seems related to job retention or termination. However, without more context, it's a bit hard for me to pinpoint exactly.

If neither of these matters to you, we could steer back towards your original interests – like pizza? Or is there something else on your mind that you'd like to discuss while we get some clarity on the mysterious "TERMINATE" word?

(And just out o

In [4]:
from typing import List, Dict, Optional
import uuid
from datetime import datetime

class OllamaEmbedding:
    # def __init__(self, base_url: str = "http://localhost:11434"):
    #     self.base_url = base_url
        
    def get_embedding(self, text: str) -> List[float]:
        response = ollama.embeddings(model="nomic-embed-text",prompt=text)
        return response['embedding']

class VectorMemory:
    def __init__(self, collection_name: str = "agent_memories"):
        self.client = chromadb.PersistentClient(path="./chroma_db")
        try:
            self.collection = self.client.get_collection(name=collection_name)
        except:
            self.collection = self.client.create_collection(name=collection_name)
        self.embedder = OllamaEmbedding()

    def create_conversation(self) -> str:
        """Create new conversation ID"""
        return str(uuid.uuid4())

    def store(self, content: str, context: str = "", conversation_id: Optional[str] = None) -> str:
        """Store message with conversation ID"""
        if not conversation_id:
            conversation_id = self.create_conversation()
            
        embedding = self.embedder.get_embedding(content)
        message_id = str(uuid.uuid4())
        
        self.collection.add(
            embeddings=[embedding],
            documents=[content],
            metadatas=[{
                "context": context,
                "conversation_id": conversation_id,
                "timestamp": str(datetime.now()),
                "message_id": message_id
            }],
            ids=[message_id]
        )
        return conversation_id

    def get_conversation(self, conversation_id: str) -> List[Dict]:
        """Retrieve all messages in a conversation"""
        results = self.collection.get(
            where={"conversation_id": conversation_id},
            include=["documents", "metadatas"]
        )
        
        # Sort by timestamp
        conversation = list(zip(results['documents'], results['metadatas']))
        conversation.sort(key=lambda x: x[1]['timestamp'])
        return conversation

    def retrieve(self, query: str, n_results: int = 3, conversation_id: Optional[str] = None) -> List[str]:
        """Retrieve similar messages, optionally filtered by conversation"""
        query_embedding = self.embedder.get_embedding(query)
        
        where_clause = {"conversation_id": conversation_id} if conversation_id else None
        
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_clause
        )
        return results['documents'][0]

In [5]:
# from vector_memory import VectorMemory

memory = VectorMemory()

# Start new conversation
conv_id = memory.store("What is quantum computing?", "Initial question")


In [6]:
conv_id

'e9267727-7c5d-486f-9600-5fa869015838'

In [7]:
# Add to conversation
memory.store("Quantum computing uses quantum phenomena...", "Response", conv_id)
memory.store("How does superposition work?", "Follow-up", conv_id)



'e9267727-7c5d-486f-9600-5fa869015838'

In [8]:
# Get entire conversation
conversation = memory.get_conversation(conv_id)
for message, metadata in conversation:
    print(f"[{metadata['timestamp']}] {message}")

# Search within conversation
similar = memory.retrieve("quantum", conversation_id=conv_id)

[2024-12-25 22:43:21.286939] What is quantum computing?
[2024-12-25 22:43:45.114489] Quantum computing uses quantum phenomena...
[2024-12-25 22:43:45.342755] How does superposition work?
